# 10_dashboard_data_prep

Pre-compute CSVs for the Streamlit dashboard so it can load instantly.
Run all cells to regenerate the files in `data/dashboard/`.

In [2]:
# Imports and paths
import pathlib, os
import pandas as pd, numpy as np
from datetime import datetime

ROOT = pathlib.Path('.')
DATA_PATH = ROOT / 'data' / 'processed' / 'ibrd_clean.csv'
OUT_DIR = ROOT / 'data' / 'dashboard'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Helper to choose column names if variants exist
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

print('Dashboard OUT_DIR:', OUT_DIR)


Dashboard OUT_DIR: data/dashboard


In [4]:
# Load data (robust to different working directories in Jupyter)
candidate_paths = [
    DATA_PATH,
    ROOT / 'data' / 'processed' / 'ibrd_clean.csv',
    pathlib.Path.cwd() / 'data' / 'processed' / 'ibrd_clean.csv',
    pathlib.Path.cwd().parent / 'data' / 'processed' / 'ibrd_clean.csv',
    pathlib.Path('/home/rigii/ATA') / 'data' / 'processed' / 'ibrd_clean.csv',
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing input data file. Checked: {candidate_paths}')

df = pd.read_csv(DATA_PATH, low_memory=False)
print('Loaded', DATA_PATH, 'shape=', df.shape)

# Map common column name variants
col_commit = pick_col(df, ['Original Principal Amount (US$)', 'Original Principal Amount', 'commitment', 'commitments'])
col_disb = pick_col(df, ['Disbursed Amount (US$)', 'Disbursed Amount', 'disbursed_amount'])
col_repaid = pick_col(df, ['Repaid to IBRD (US$)', 'Repaid to IBRD', 'repaid'])
col_out = pick_col(df, ['Due to IBRD (US$)', 'Due to IBRD', 'outstanding', 'outstanding_amount'])
col_loan = pick_col(df, ['Loan Number', 'Loan_Number', 'loan_number'])
col_country = pick_col(df, ['Country / Economy', 'Country', 'country'])
col_region = pick_col(df, ['Region', 'region'])
col_status = pick_col(df, ['Loan Status', 'Loan_Status', 'status'])
col_year = pick_col(df, ['Approval Year', 'approval_year', 'Approval_Year', 'approvalYear'])
col_risk = pick_col(df, ['risk_score', 'Risk Score', 'loan_risk_score', 'risk'])
col_size = pick_col(df, ['loan_size_category', 'Loan Size Category', 'loan_size'])
col_age = pick_col(df, ['loan_age_years', 'loan_age', 'age_years'])
print('Mapped cols:', dict(commit=col_commit, disb=col_disb, repaid=col_repaid, out=col_out, loan=col_loan, country=col_country, region=col_region, status=col_status, year=col_year, risk=col_risk, size=col_size, age=col_age))

Loaded /home/rigii/ATA/data/processed/ibrd_clean.csv shape= (9518, 54)
Mapped cols: {'commit': 'Original Principal Amount (US$)', 'disb': 'Disbursed Amount (US$)', 'repaid': 'Repaid to IBRD (US$)', 'out': 'Due to IBRD (US$)', 'loan': 'Loan Number', 'country': 'Country / Economy', 'region': 'Region', 'status': 'Loan Status', 'year': 'approval_year', 'risk': 'risk_score', 'size': 'loan_size_category', 'age': 'loan_age_years'}


## Files produced
- `kpi_summary.csv`: Overall KPIs for the portfolio
- `regional_summary.csv`: Aggregated by Region
- `country_summary.csv`: Aggregated by Country
- `yearly_summary.csv`: Aggregated by approval year
- `status_summary.csv`: Aggregated by loan status
- `loan_size_summary.csv`: Aggregated by loan size category
- `age_summary.csv`: Aggregated by loan age category
- `top_countries.csv`: Top 20 countries by commitments
- `high_risk_summary.csv`: High-risk loans grouped by region/country
- `forecast_data.csv`: Historical + simple forecast commitments by year

In [15]:
# 1) KPI summary
commitments = df[col_commit].sum() if col_commit else 0
disbursed = df[col_disb].sum() if col_disb else 0
repaid = df[col_repaid].sum() if col_repaid else 0
outstanding = df[col_out].sum() if col_out else (commitments - repaid)
num_loans = df[col_loan].nunique() if col_loan else len(df)
num_countries = df[col_country].nunique() if col_country else 0
num_regions = df[col_region].nunique() if col_region else 0
repayment_rate = (repaid / commitments) if commitments else np.nan
cancellations = 0
if col_status and 'cancel' in ' '.join(df[col_status].dropna().astype(str).str.lower().unique()):
    cancellations = (df[col_status].str.lower().str.contains('cancel').sum())
cancellation_rate = (cancellations / num_loans) if num_loans else 0
avg_loan_size = (df[col_commit].mean() if col_commit else np.nan)
median_loan_size = (df[col_commit].median() if col_commit else np.nan)

kpi = pd.DataFrame([{
    'total_commitments': commitments,
    'total_disbursed': disbursed,
    'total_repaid': repaid,
    'total_outstanding': outstanding,
    'num_loans': num_loans,
    'num_countries': num_countries,
    'num_regions': num_regions,
    'repayment_rate': repayment_rate,
    'avg_loan_size': avg_loan_size,
    'median_loan_size': median_loan_size,
    'cancellations': cancellations,
    'cancellation_rate': cancellation_rate
}])

out_path = OUT_DIR / 'kpi_summary.csv'
kpi.to_csv(out_path, index=False)
print('Saved', out_path, 'shape=', kpi.shape)
print(kpi.head(5).to_string(index=False))

Saved data/dashboard/kpi_summary.csv shape= (1, 12)
 total_commitments  total_disbursed  total_repaid  total_outstanding  num_loans  num_countries  num_regions  repayment_rate  avg_loan_size  median_loan_size  cancellations  cancellation_rate
      9.955982e+11     7.665112e+11  4.651577e+11       2.984153e+11       9518            148            7        0.467214   1.046016e+08        40000000.0            220           0.023114


In [25]:
# 2) regional_summary.csv
if col_region:
    grp = df.groupby(col_region).agg(
        commitments=(col_commit if col_commit else df.columns[0],'sum'),
        loans=(col_loan if col_loan else df.columns[0],'nunique'),
        outstanding=(col_out if col_out else df.columns[0],'sum'),
        avg_risk=(col_risk if col_risk else df.columns[0],'mean'),
        repaid=(col_repaid if col_repaid else df.columns[0],'sum')
    ).reset_index()
    grp['repayment_rate'] = grp['repaid'] / grp['commitments']
    out_path = OUT_DIR / 'regional_summary.csv'
    grp.to_csv(out_path, index=False)
    print('Saved', out_path, 'shape=', grp.shape)
    print(grp.head())
else:
    print('No region column found; skipping regional_summary')

Saved data/dashboard/regional_summary.csv shape= (7, 7)
                          Region   commitments  loans   outstanding  avg_risk  \
0          EAST ASIA AND PACIFIC  2.078934e+11   2014  5.531646e+10  0.154841   
1    EASTERN AND SOUTHERN AFRICA  3.174331e+10    438  1.559403e+10  0.185731   
2        EUROPE AND CENTRAL ASIA  2.170593e+11   1948  6.725453e+10  0.222562   
3    LATIN AMERICA AND CARIBBEAN  3.061429e+11   2975  8.757018e+10  0.165176   
4  MID EAST,NORTH AFRICA,AFG,PAK  1.190767e+11   1256  4.418930e+10  0.181409   

         repaid  repayment_rate  
0  1.092499e+11        0.525509  
1  6.003578e+09        0.189129  
2  9.612828e+10        0.442866  
3  1.596821e+11        0.521593  
4  4.575127e+10        0.384217  


In [34]:
# 3) country_summary.csv
if col_country:
    cgrp = df.groupby(col_country).agg(
        commitments=(col_commit if col_commit else df.columns[0],'sum'),
        loans=(col_loan if col_loan else df.columns[0],'nunique'),
        outstanding=(col_out if col_out else df.columns[0],'sum'),
        avg_risk=(col_risk if col_risk else df.columns[0],'mean')
    ).reset_index()
    if 'cluster' in df.columns:
        cgrp = cgrp.merge(df[[col_country,'cluster']].drop_duplicates(col_country), left_on=col_country, right_on=col_country, how='left')
    out_path = OUT_DIR / 'country_summary.csv'
    cgrp.to_csv(out_path, index=False)
    print('Saved', out_path, 'shape=', cgrp.shape)
    print(cgrp.head())
else:
    print('No country column found; skipping country_summary')

Saved data/dashboard/country_summary.csv shape= (148, 5)
     Country / Economy   commitments  loans   outstanding  avg_risk
0              Albania  2.345510e+09     41  1.237265e+09  0.574390
1              Algeria  5.911830e+09    128  0.000000e+00  0.059375
2               Angola  1.003100e+10     37  5.926827e+09  0.631081
3  Antigua And Barbuda  1.200000e+07      2  3.719059e+06  0.700000
4            Argentina  4.824665e+10    295  1.271579e+10  0.233220


In [35]:
# 4) yearly_summary.csv
if col_year:
    ygrp = df.groupby(col_year).agg(
        loans=(col_loan if col_loan else df.columns[0],'nunique'),
        commitments=(col_commit if col_commit else df.columns[0],'sum'),
        disbursed=(col_disb if col_disb else df.columns[0],'sum'),
        repaid=(col_repaid if col_repaid else df.columns[0],'sum')
    ).reset_index().sort_values(col_year)
    out_path = OUT_DIR / 'yearly_summary.csv'
    ygrp.to_csv(out_path, index=False)
    print('Saved', out_path, 'shape=', ygrp.shape)
    print(ygrp.head())
else:
    print('No approval year column; skipping yearly_summary')

Saved data/dashboard/yearly_summary.csv shape= (80, 5)
   approval_year  loans  commitments     disbursed        repaid
0           1947      5  497000000.0  4.967620e+08  1.228012e+08
1           1948      8   28000000.0  2.800000e+07  1.292200e+07
2           1949     12  219145000.0  1.973284e+08  1.354683e+08
3           1950     17  279230000.0  2.611484e+08  1.949746e+08
4           1951     17  208408000.0  2.074416e+08  1.110384e+08


In [36]:
# 5) status_summary.csv
if col_status:
    sgrp = df.groupby(col_status).agg(
        count=(col_loan if col_loan else df.columns[0],'nunique'),
        total_amount=(col_commit if col_commit else df.columns[0],'sum')
    ).reset_index()
    total_port = sgrp['total_amount'].sum()
    sgrp['pct_portfolio'] = sgrp['total_amount'] / total_port
    out_path = OUT_DIR / 'status_summary.csv'
    sgrp.to_csv(out_path, index=False)
    print('Saved', out_path, 'shape=', sgrp.shape)
    print(sgrp.head())
else:
    print('No status column; skipping status_summary')

Saved data/dashboard/status_summary.csv shape= (11, 4)
           Loan Status  count  total_amount  pct_portfolio
0             Approved     71  2.150861e+10       0.021604
1           Disbursing    531  1.234518e+11       0.123998
2  Disbursing&Repaying    203  3.795231e+10       0.038120
3            Effective     61  1.763139e+10       0.017709
4      Fully Cancelled    220  1.558452e+10       0.015653


In [37]:
# 6) loan_size_summary.csv
if col_size:
    lgrp = df.groupby(col_size).agg(
        count=(col_loan if col_loan else df.columns[0],'nunique'),
        avg_commitment=(col_commit if col_commit else df.columns[0],'mean'),
        avg_risk=(col_risk if col_risk else df.columns[0],'mean')
    ).reset_index()
    out_path = OUT_DIR / 'loan_size_summary.csv'
    lgrp.to_csv(out_path, index=False)
    print('Saved', out_path, 'shape=', lgrp.shape)
    print(lgrp.head())
else:
    print('No loan size category column; skipping loan_size_summary')

Saved data/dashboard/loan_size_summary.csv shape= (4, 4)
  loan_size_category  count  avg_commitment  avg_risk
0              Large   1141    2.880523e+08  0.368361
1             Medium   2832    9.796862e+07  0.208775
2               Mega    400    7.443656e+08  0.479625
3              Small   5145    1.783036e+07  0.102439


In [38]:
# 7) age_summary.csv (create age buckets if needed)
if col_age:
    df['age_category'] = pd.cut(df[col_age].fillna(-1), bins=[-1,0,2,5,10,100], labels=['Unknown','0-2','3-5','6-10','10+'])
    agr = df.groupby('age_category').agg(count=(col_loan if col_loan else df.columns[0],'nunique'), outstanding=(col_out if col_out else df.columns[0],'sum'), avg_repayment=(col_repaid if col_repaid else df.columns[0],'mean')).reset_index()
    out_path = OUT_DIR / 'age_summary.csv'
    agr.to_csv(out_path, index=False)
    print('Saved', out_path, 'shape=', agr.shape)
    print(agr.head())
else:
    print('No age column; skipping age_summary')

Saved data/dashboard/age_summary.csv shape= (5, 4)
  age_category  count   outstanding  avg_repayment
0      Unknown      1  0.000000e+00   0.000000e+00
1          0-2    316  2.839778e+10   4.252132e+04
2          3-5    431  6.937503e+10   2.657062e+06
3         6-10    651  8.930256e+10   1.625505e+07
4          10+   8119  1.113400e+11   5.584641e+07


/tmp/ipykernel_1794333/1465982195.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agr = df.groupby('age_category').agg(count=(col_loan if col_loan else df.columns[0],'nunique'), outstanding=(col_out if col_out else df.columns[0],'sum'), avg_repayment=(col_repaid if col_repaid else df.columns[0],'mean')).reset_index()


In [39]:
# 8) top_countries.csv
if col_country:
    top = df.groupby(col_country).agg(commitments=(col_commit if col_commit else df.columns[0],'sum')).reset_index().sort_values('commitments', ascending=False).head(20)
    out_path = OUT_DIR / 'top_countries.csv'
    top.to_csv(out_path, index=False)
    print('Saved', out_path, 'shape=', top.shape)
    print(top.head())
else:
    print('No country column; skipping top_countries')

Saved data/dashboard/top_countries.csv shape= (20, 2)
    Country / Economy   commitments
61              India  9.089926e+10
19             Brazil  7.601575e+10
62          Indonesia  7.244698e+10
136           Turkiye  6.391224e+10
87             Mexico  6.227015e+10


In [40]:
# 9) high_risk_summary.csv (top risk loans grouped)
if col_risk:
    thresh = df[col_risk].quantile(0.9)
    high = df[df[col_risk] >= thresh].copy()
    group_cols = []
    if col_region: group_cols.append(col_region)
    if col_country: group_cols.append(col_country)
    if not group_cols: group_cols = [col_country] if col_country else [col_region]
    hr = high.groupby(group_cols).agg(count=(col_loan if col_loan else df.columns[0],'nunique'), total_outstanding=(col_out if col_out else df.columns[0],'sum')).reset_index()
    out_path = OUT_DIR / 'high_risk_summary.csv'
    hr.to_csv(out_path, index=False)
    print('Saved', out_path, 'shape=', hr.shape)
    print(hr.head())
else:
    print('No risk column; skipping high_risk_summary')

Saved data/dashboard/high_risk_summary.csv shape= (93, 4)
                  Region   Country / Economy  count  total_outstanding
0  EAST ASIA AND PACIFIC               China    110       8.823849e+09
1  EAST ASIA AND PACIFIC                Fiji      6       2.586107e+08
2  EAST ASIA AND PACIFIC           Indonesia     77       1.395687e+10
3  EAST ASIA AND PACIFIC  Korea, Republic Of      2       0.000000e+00
4  EAST ASIA AND PACIFIC            Malaysia      2       0.000000e+00


In [41]:
# 10) forecast_data.csv (historical + naive forecast)
if col_year and col_commit:
    hist = df.groupby(col_year).agg(commitments=(col_commit,'sum')).reset_index().sort_values(col_year)
    hist = hist.rename(columns={col_year: 'year'})
    hist['year'] = hist['year'].astype(int)
    hist['growth'] = hist['commitments'].pct_change()
    mean_growth = hist['growth'].dropna().mean() if not hist['growth'].dropna().empty else 0
    last_year = int(hist['year'].max()) if not hist.empty else datetime.now().year
    future_years = [last_year + i for i in range(1,6)]
    last_val = float(hist['commitments'].iloc[-1]) if not hist.empty else 0
    forecasts = []
    val = last_val
    for y in future_years:
        forecasts.append({'year': int(y), 'commitments': float(val)})
    fdf = pd.DataFrame(forecasts)
    outdf = pd.concat([hist[['year','commitments']], fdf], ignore_index=True)
    out_path = OUT_DIR / 'forecast_data.csv'
    outdf.to_csv(out_path, index=False)
    print('Saved', out_path, 'shape=', outdf.shape)
    print(outdf.tail())
else:
    print('Missing year or commitments column; skipping forecast_data')

Saved data/dashboard/forecast_data.csv shape= (85, 2)
    year   commitments
80  2027  4.013284e+10
81  2028  4.013284e+10
82  2029  4.013284e+10
83  2030  4.013284e+10
84  2031  4.013284e+10


---
All files are saved to `data/dashboard/`. Each cell prints the saved file path, shape, and a small preview. Run cells top-to-bottom to refresh all artifacts.